BOOK RECOMMENDATION ENGINE

Import Libraries

In [46]:
import pandas as pd
import numpy as np
import pickle

Load Dataset

In [47]:
books = pd.read_csv("books.xls")

Data Preprocessing

In [48]:
books.head()

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


In [49]:
books.tail()

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
6805,9788185300535,8185300534,I Am that,Talks with Sri Nisargadatta Maharaj,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0
6806,9788185944609,8185944601,Secrets Of The Heart,NaN,Khalil Gibran,Mysticism,http://books.google.com/books/content?id=XcrVp...,NaN,1993.0,4.08,74.0,324.0
6807,9788445074879,8445074873,Fahrenheit 451,NaN,Ray Bradbury,Book burning,NaN,NaN,2004.0,3.98,186.0,5733.0
6808,9789027712059,9027712050,The Berlin Phenomenology,NaN,Georg Wilhelm Friedrich Hegel,History,http://books.google.com/books/content?id=Vy7Sk...,Since the three volume edition ofHegel's Philo...,1981.0,0.00,210.0,0.0
6809,9789042003408,9042003405,'I'm Telling You Stories',Jeanette Winterson and the Politics of Reading,Helena Grice;Tim Woods,Literary Criticism,http://books.google.com/books/content?id=2lVyR...,This is a jubilant and rewarding collection of...,1998.0,3.70,136.0,10.0


In [50]:
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6810 entries, 0 to 6809
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   isbn13          6810 non-null   int64  
 1   isbn10          6810 non-null   object 
 2   title           6810 non-null   object 
 3   subtitle        2381 non-null   object 
 4   authors         6738 non-null   object 
 5   categories      6711 non-null   object 
 6   thumbnail       6481 non-null   object 
 7   description     6548 non-null   object 
 8   published_year  6804 non-null   float64
 9   average_rating  6767 non-null   float64
 10  num_pages       6767 non-null   float64
 11  ratings_count   6767 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 638.6+ KB


In [51]:
books.describe()

,isbn13,published_year,average_rating,num_pages,ratings_count
count,6.810000e+03,6804.000000,6767.000000,6767.000000,6.767000e+03
mean,9.780677e+12,1998.630364,3.933284,348.181026,2.106910e+04
std,6.068911e+08,10.484257,0.331352,242.376783,1.376207e+05
min,9.780002e+12,1853.000000,0.000000,0.000000,0.000000e+00
25%,9.780330e+12,1996.000000,3.770000,208.000000,1.590000e+02
50%,9.780553e+12,2002.000000,3.960000,304.000000,1.018000e+03
75%,9.780810e+12,2005.000000,4.130000,420.000000,5.992500e+03
max,9.789042e+12,2019.000000,5.000000,3342.000000,5.629932e+06


In [52]:
books.shape

(6810, 12)

In [53]:
books.size

81720

In [54]:
books.columns

Index(['isbn13', 'isbn10', 'title', 'subtitle', 'authors', 'categories',
       'thumbnail', 'description', 'published_year', 'average_rating',
       'num_pages', 'ratings_count'],
      dtype='object')

Select Required Columns

In [37]:
books = books[['title', 'authors', 'categories', 'description', 'thumbnail']]

Handle Missing Values

In [38]:
books.fillna('', inplace=True)

Create Tags

In [39]:
books['tags'] = (
    books['title'] + " " +
    books['authors'] + " " +
    books['categories'] + " " +
    books['description']
)

books['tags'] = books['tags'].str.lower()

Convert Text → Vectors (TF-IDF)

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=8000,
    stop_words='english'
)

vectors = tfidf.fit_transform(books['tags']).toarray()

Compute Similarity

In [41]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

Save Files

In [42]:
pickle.dump(books.to_dict(), open('books.pkl', 'wb'))
pickle.dump(similarity, open('similarity_books.pkl', 'wb'))

Recommendation Function

In [43]:
def recommend(book):
    if book not in books['title'].values:
        return [], [], []

    book_index = books[books['title'] == book].index[0]
    distances = similarity[book_index]

    books_list = sorted(
        enumerate(distances),
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    names = []
    authors = []
    posters = []

    for i in books_list:
        names.append(books.iloc[i[0]]['title'])
        authors.append(books.iloc[i[0]]['authors'])
        posters.append(books.iloc[i[0]]['thumbnail'])

    return names, authors, posters

TESTING

In [44]:
sample_book = "Gilead"

print("Selected Book:", sample_book)

names, authors, posters = recommend(sample_book)

print("\nRecommended Books:\n")

for i in range(len(names)):
    print(f"{i+1}. {names[i]} by {authors[i]}")

Selected Book: Gilead

Recommended Books:

1. The Handmaid's Tale by Margaret Atwood
2. Four Baboons Adoring the Sun by John Guare
3. The Languages of Pao by Jack Vance
4. Laguna, I Love You by John Weld
5. Go Tell it on the Mountain by James Baldwin


In [45]:
from IPython.display import Image, display

for i in range(len(names)):
    print(names[i])
    display(Image(url=posters[i]))

The Handmaid's Tale


Four Baboons Adoring the Sun


The Languages of Pao


Laguna, I Love You


Go Tell it on the Mountain
